# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/titlyzaman25/flyrank-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
!git clone https://github.com/titlyzaman25/flyrank-internship.git

Cloning into 'flyrank-internship'...
remote: Enumerating objects: 269, done.
remote: Counting objects: 100% (269/269), done.
remote: Compressing objects: 100% (207/207), done.
remote: Total 269 (delta 139), reused 126 (delta 44), pack-reused 0 (from 0)
Receiving objects: 100% (269/269), 2.16 MiB | 16.24 MiB/s, done.
Resolving deltas: 100% (139/139), done.


In [18]:
%cd /content/flyrank-internship
!ls -lah
!ls -lah data/raw

/content/flyrank-internship
total 108K
drwxr-xr-x 13 root root 4.0K Aug 30 07:50 .
drwxr-xr-x  1 root root 4.0K Aug 30 07:48 ..
-rw-r--r--  1 root root  654 Aug 30 07:48 AGENTS.md
-rw-r--r--  1 root root  654 Aug 30 07:48 CLAUDE.md
drwxr-xr-x  3 root root 4.0K Aug 30 07:48 data
-rw-r--r--  1 root root 2.7K Aug 30 07:48 DATA_USE.md
drwxr-xr-x  3 root root 4.0K Aug 30 07:48 docs
drwxr-xr-x 12 root root 4.0K Aug 30 07:50 flyrank-internship
drwxr-xr-x  8 root root 4.0K Aug 30 07:48 .git
drwxr-xr-x  3 root root 4.0K Aug 30 07:48 .github
-rw-r--r--  1 root root  993 Aug 30 07:48 .gitignore
-rw-r--r--  1 root root  11K Aug 30 07:48 GUIDE.md
-rw-r--r--  1 root root 1.3K Aug 30 07:48 LICENSE
drwxr-xr-x  2 root root 4.0K Aug 30 07:48 notebooks
drwxr-xr-x  3 root root 4.0K Aug 30 07:48 outputs
-rw-r--r--  1 root root 9.6K Aug 30 07:48 README.md
-rw-r--r--  1 root root  107 Aug 30 07:48 requirements.txt
drwxr-xr-x  2 root root 4.0K Aug 30 07:48 scripts
-rw-r--r--  1 root root 5.6K Aug 30 07:48 SET

In [19]:
import os

for root, dirs, files in os.walk("/content/flyrank-internship/data"):
    for file in files:
        print(os.path.join(root, file))

/content/flyrank-internship/data/raw/content_refresh_anonymized.csv


In [20]:
import pandas as pd
import numpy as np
df = pd.read_csv("/content/flyrank-internship/data/raw/content_refresh_anonymized.csv")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")

Rows: 30,000 | Columns: 44


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*
Looking at the shape of key fields before testing anything — percentiles instead of just
the mean, since these fields have heavy tails (a few pages with huge impressions or CTR
values can distort a mean badly).

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
key_fields = ["impressions_90d", "clicks_90d", "ctr", "engagement_rate",
              "content_age_days", "word_count", "scroll_rate"]

for field in key_fields:
    print(f"--- {field} ---")
    print(df[field].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).round(2))
    print()

print("Heavy-tail check: max vs 99th percentile")
for field in ["impressions_90d", "clicks_90d", "scroll_rate"]:
    p99 = df[field].quantile(0.99)
    mx = df[field].max()
    print(f"{field:20s} | p99: {p99:,.2f} | max: {mx:,.2f} | ratio: {mx/p99 if p99 else float('nan'):.1f}x")

print("\navg_position = 0 count (means 'no data', not rank zero — per the data dictionary):")
print((df["avg_position"] == 0).sum())

--- impressions_90d ---
count     30000.00
mean       5200.37
std       16838.02
min           1.00
10%           5.00
25%          81.00
50%         731.00
75%        3615.25
90%       12136.40
99%       73505.83
max      517715.00
Name: impressions_90d, dtype: float64

--- clicks_90d ---
count    30000.00
mean        16.10
std         75.08
min          0.00
10%          0.00
25%          0.00
50%          1.00
75%          7.00
90%         32.00
99%        253.01
max       4178.00
Name: clicks_90d, dtype: float64

--- ctr ---
count    30000.00
mean         0.51
std          3.28
min          0.00
10%          0.00
25%          0.00
50%          0.07
75%          0.29
90%          0.65
99%          8.33
max        100.00
Name: ctr, dtype: float64

--- engagement_rate ---
count    30000.00
mean         2.53
std          8.31
min          0.00
10%          0.00
25%          0.00
50%          0.00
75%          1.35
90%          6.94
99%         33.33
max        100.00
Name: engagement_r

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*
Three safe, observable signals tested against declining/growing status.

**Signal #1: Content age vs trend direction**
Hypothesis: older content is more likely to be declining.
**Signal #2: Word count vs CTR**
Hypothesis: longer content earns a higher CTR.
**Signal #3: Scroll rate vs engagement rate**
Hypothesis: pages with higher scroll rate also show higher engagement rate (same underlying reader behavior).

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal #1: content age vs trend
age_by_trend = df.groupby("trend_direction")["content_age_days"].mean().round(1)
print("Mean content age by trend direction:")
print(age_by_trend)
print(f"\nVerdict: {'CONFIRMED' if age_by_trend.get('down', 0) > age_by_trend.get('up', 0) else 'OPPOSITE'} — "
      f"declining content averages {age_by_trend.get('down', float('nan')):.0f} days vs "
      f"{age_by_trend.get('up', float('nan')):.0f} days for growing content.")
# Signal #2: word count vs CTR, using terciles to avoid noise from outlier pages
df["word_count_tier"] = pd.qcut(df["word_count"], 3, labels=["short", "medium", "long"], duplicates="drop")
ctr_by_length = df.groupby("word_count_tier", observed=True)["ctr"].mean().round(3)
print("Mean CTR by word count tier:")
print(ctr_by_length)
spread = ctr_by_length.max() - ctr_by_length.min()
print(f"\nVerdict: {'MIXED' if spread < 0.05 else 'CONFIRMED'} — spread of {spread:.3f} percentage points "
      f"across tiers is {'small' if spread < 0.05 else 'notable'}.")
# Signal #3: scroll rate vs engagement rate correlation
corr = df[["scroll_rate", "engagement_rate"]].corr().iloc[0, 1]
print(f"Correlation (scroll_rate, engagement_rate): {corr:.3f}")
verdict = "CONFIRMED" if corr > 0.3 else ("MIXED" if corr > 0.1 else "FALSE")
print(f"Verdict: {verdict} — {'a real positive relationship' if corr > 0.3 else 'a weak or negligible relationship'} in this data.")

Mean content age by trend direction:
trend_direction
down      236.2
flat      245.9
new       238.7
stable    295.4
up        288.5
Name: content_age_days, dtype: float64

Verdict: OPPOSITE — declining content averages 236 days vs 288 days for growing content.
Mean CTR by word count tier:
word_count_tier
short     1.107
medium    0.448
long      0.242
Name: ctr, dtype: float64

Verdict: CONFIRMED — spread of 0.865 percentage points across tiers is notable.
Correlation (scroll_rate, engagement_rate): 0.163
Verdict: MIXED — a weak or negligible relationship in this data.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*
Testing the assumption behind the "Fix CTR" flag: that flagged pages actually have a
CTR below what their position would predict.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Does avg_position meaningfully predict expected CTR, the assumption "Fix CTR" flags rely on?
valid = df[df["avg_position"] > 0].copy()  # exclude "no data" rows
valid["position_bucket"] = pd.cut(
    valid["avg_position"],
    bins=[0, 3, 10, 20, 50, 1000],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"]
)

ctr_by_position = valid.groupby("position_bucket", observed=True)["ctr"].mean().round(3)
print("Mean CTR by position bucket:")
print(ctr_by_position)

is_monotonic_decreasing = ctr_by_position.is_monotonic_decreasing
print(f"\nVerdict: {'CONFIRMED' if is_monotonic_decreasing else 'MIXED'} — CTR "
      f"{'decreases consistently' if is_monotonic_decreasing else 'does not decrease consistently'} "
      f"as position gets worse, which is the core assumption a position-based CTR flag relies on.")

# Flag-specific check, if a flag column exists in this export
if "optimization_flag" in df.columns:
    print("\nCTR by optimization_flag value:")
    print(df.groupby("optimization_flag")["ctr"].mean().round(3))

Mean CTR by position bucket:
position_bucket
1-3      2.714
4-10     0.651
11-20    0.323
21-50    0.222
50+      0.151
Name: ctr, dtype: float64

Verdict: CONFIRMED — CTR decreases consistently as position gets worse, which is the core assumption a position-based CTR flag relies on.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Position remains the most reliable of these signals — CTR drops consistently as position
worsens, which supports using position-based expected-CTR rules as a baseline. Content age
and word count are weaker and noisier signals on their own; they're worth keeping as
features but shouldn't be trusted as standalone triggers for action. A content team should
treat these as directional inputs to combine with other signals, not single-variable rules
to act on in isolation.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.